# 로컬 벤치마크 (제출 전 점수 추정)

## 대회 평가 기준 재현
- **PerfNorm** = GSM8K exact_match (flexible-extract, 5-shot, 512 samples) / 0.6484
- **SpeedNorm** = 1 - (base_time_per_token / model_time_per_token)
- **Score** = max(0.5 × PerfNorm + 0.5 × SpeedNorm, 0)

### 실행 전 설정
1. **런타임 → 런타임 유형 변경 → GPU (T4)** 선택
2. **하이RAM** 옵션 활성화

### 주의사항
- T4 ≠ L4 이므로 SpeedNorm은 추정치 (상대 비교는 유효)
- PerfNorm은 GPU 무관하게 정확히 재현 가능
- 여러 모델을 비교하여 제출 전 최적 모델 선택에 활용

---

# 0. 환경 설정 (Colab 전용)

In [ ]:
# ============================================================================
# Colab 패키지 설치
# ============================================================================
!pip install -q numpy==1.26.4
!pip install -q transformers==4.57.3 datasets torch accelerate
!pip install -q compressed-tensors==0.13.0
!pip install -q vllm
!pip install -q lm-eval

print("\n⚠️ 설치 완료! 런타임 → 런타임 다시 시작 후 다음 셀부터 실행하세요")

In [ ]:
# ============================================================================
# 베이스 모델 다운로드 (PerfNorm/SpeedNorm 기준선)
# ============================================================================
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="LGAI-EXAONE/EXAONE-4.0-1.2B",
    local_dir="./base_model",
)

print("\u2705 베이스 모델 다운로드 완료!")

In [ ]:
# ============================================================================
# 테스트할 양자화 모델 업로드
# ============================================================================
# 방법 1: submit zip 업로드 후 압축 해제
# 방법 2: 이 노트북에서 직접 양자화한 ./model 사용

import os
import zipfile

# zip 파일이 있으면 압축 해제
USE_ZIP = False  # True로 변경하면 zip 업로드 모드

if USE_ZIP:
    from google.colab import files
    print("submit zip 파일을 업로드하세요...")
    uploaded = files.upload()
    
    zip_name = list(uploaded.keys())[0]
    print(f"[INFO] {zip_name} 압축 해제 중...")
    
    with zipfile.ZipFile(zip_name, 'r') as zf:
        zf.extractall(".")
    
    # zip 내부 구조 확인 (model/ 디렉토리)
    if os.path.exists("./model"):
        print("\u2705 ./model 디렉토리 확인")
    else:
        # zip 내부 경로가 다를 수 있음
        for d in os.listdir("."):
            if os.path.isdir(d) and "model" in d.lower():
                print(f"  발견: ./{d}/")
else:
    print("[INFO] ./model 디렉토리의 모델을 사용합니다")
    if os.path.exists("./model"):
        print("\u2705 ./model 존재")
        for f in sorted(os.listdir("./model"))[:5]:
            print(f"  {f}")
    else:
        print("\u274c ./model 없음 - 먼저 양자화 노트북을 실행하거나 USE_ZIP=True로 변경")

# 1. Import + GPU 확인

In [ ]:
import os
import time
import json
import torch
import subprocess

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({vram:.1f} GB)")
    print(f"\n\u26a0\ufe0f T4 GPU에서 측정한 SpeedNorm은 L4와 다를 수 있습니다")
    print("  → PerfNorm은 정확, SpeedNorm은 상대 비교용")

import psutil
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
print("\n\u2705 준비 완료!")

# 2. 설정

In [ ]:
# ============================================================================
# ⭐ 테스트할 모델 경로
# ============================================================================
MODEL_PATH = "./model"          # 양자화 모델 경로
BASE_MODEL_PATH = "./base_model" # 원본 FP16 모델 경로

# ============================================================================
# 대회 평가 설정 (변경 금지)
# ============================================================================
BASE_PERF = 0.6484  # 원본 모델 GSM8K exact_match (flexible-extract)
GSM8K_LIMIT = 512   # 평가 샘플 수
GPU_MEM_UTIL = 0.85 # vLLM gpu_memory_utilization

# ============================================================================
# 빠른 테스트 모드 (전체 실행 전 빠르게 확인)
# ============================================================================
QUICK_MODE = False   # True: 64 samples로 빠르게 / False: 512 samples 전체

if QUICK_MODE:
    EVAL_LIMIT = 64
    print("\u26a1 QUICK MODE: 64 samples (약 2-3분)")
    print("  → 정확도는 낮지만 빠른 비교에 유용")
else:
    EVAL_LIMIT = GSM8K_LIMIT
    print(f"\U0001f3af FULL MODE: {GSM8K_LIMIT} samples (약 10-15분)")
    print("  → 대회와 동일한 평가")

print(f"\n모델 경로: {MODEL_PATH}")
print(f"베이스 모델: {BASE_MODEL_PATH}")
print(f"평가 샘플: {EVAL_LIMIT}")

---

# 3. PerfNorm 측정 (GSM8K)

대회와 동일한 lm-evaluation-harness + vLLM 사용.

In [ ]:
def run_lm_eval(model_path, limit, output_name="results"):
    """lm_eval로 GSM8K 성능 측정 (대회 동일 설정)."""
    output_path = f"./eval_{output_name}"
    
    cmd = [
        "lm_eval",
        "--model", "vllm",
        "--model_args", f"pretrained={model_path},gpu_memory_utilization={GPU_MEM_UTIL},enable_thinking=False,max_gen_toks=2048,trust_remote_code=True",
        "--tasks", "gsm8k",
        "--num_fewshot", "5",
        "--limit", str(limit),
        "--output_path", output_path,
        "--apply_chat_template",
        "--batch_size", "auto",
    ]
    
    print(f"[INFO] lm_eval 실행: {model_path}")
    print(f"  명령어: {' '.join(cmd)}")
    print(f"  대기 중...\n")
    
    start_time = time.time()
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=1200,  # 20분 타임아웃
    )
    
    elapsed = time.time() - start_time
    
    if result.returncode != 0:
        print(f"\u274c lm_eval 실패!")
        print(f"STDERR: {result.stderr[-2000:]}")
        return None, elapsed
    
    # 결과 파싱
    print(result.stdout[-3000:])
    
    # JSON 결과 파일 찾기
    score = None
    for root, dirs, files in os.walk(output_path):
        for f in files:
            if f.endswith(".json") and "results" in f:
                fpath = os.path.join(root, f)
                with open(fpath) as jf:
                    data = json.load(jf)
                
                # GSM8K 결과 추출
                if "results" in data:
                    for task_key, task_data in data["results"].items():
                        if "gsm8k" in task_key:
                            # flexible-extract exact_match 찾기
                            for metric_key, metric_val in task_data.items():
                                if "exact_match" in metric_key and "flexible" in metric_key:
                                    score = metric_val
                                    print(f"\n\u2705 GSM8K exact_match (flexible-extract): {score}")
                                    break
                            # flexible 못 찾으면 일반 exact_match
                            if score is None:
                                for metric_key, metric_val in task_data.items():
                                    if "exact_match" in metric_key:
                                        score = metric_val
                                        print(f"\n\u2705 GSM8K exact_match: {score}")
                                        break
    
    if score is None:
        # stdout에서 직접 파싱 시도
        for line in result.stdout.split("\n"):
            if "exact_match" in line and ("flexible" in line or "gsm8k" in line.lower()):
                parts = line.split("|")
                for part in parts:
                    part = part.strip()
                    try:
                        val = float(part)
                        if 0 <= val <= 1:
                            score = val
                            break
                    except ValueError:
                        continue
    
    return score, elapsed

print("\u2705 lm_eval 함수 정의 완료")

In [ ]:
# ============================================================================
# 양자화 모델 PerfNorm 측정
# ============================================================================
print("=" * 60)
print("양자화 모델 GSM8K 평가")
print("=" * 60)

model_score, model_time = run_lm_eval(MODEL_PATH, EVAL_LIMIT, "model")

if model_score is not None:
    perf_norm = model_score / BASE_PERF
    print(f"\n{'='*40}")
    print(f"  GSM8K exact_match: {model_score:.4f}")
    print(f"  PerfNorm: {model_score:.4f} / {BASE_PERF} = {perf_norm:.4f}")
    print(f"  소요 시간: {model_time:.1f}초 ({model_time/60:.1f}분)")
    print(f"{'='*40}")
else:
    perf_norm = None
    print("\u274c PerfNorm 측정 실패 - 아래 수동 측정 셀 사용")

## 3-1. PerfNorm 수동 측정 (lm_eval 실패 시)

vLLM이 안 되는 경우 HuggingFace 모델로 직접 측정.

In [ ]:
# lm_eval이 실패한 경우에만 실행
MANUAL_PERF = False  # True로 변경하여 수동 측정

if MANUAL_PERF:
    print("[INFO] HuggingFace 모델로 GSM8K 수동 측정...")
    
    # lm_eval with hf backend
    cmd = [
        "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={MODEL_PATH},trust_remote_code=True",
        "--tasks", "gsm8k",
        "--num_fewshot", "5",
        "--limit", str(EVAL_LIMIT),
        "--output_path", "./eval_model_hf",
        "--apply_chat_template",
        "--batch_size", "4",
    ]
    
    print(f"  명령어: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(f"STDERR: {result.stderr[-2000:]}")
else:
    print("[INFO] 수동 측정 건너뜀 (MANUAL_PERF=False)")

---

# 4. SpeedNorm 측정 (vLLM 추론 속도)

원본 모델과 양자화 모델의 토큰 생성 속도 비교.

In [ ]:
def measure_throughput(model_path, num_prompts=20, max_tokens=256):
    """vLLM으로 추론 속도 측정 (tokens/sec)."""
    from vllm import LLM, SamplingParams
    
    print(f"[INFO] vLLM 로드: {model_path}")
    llm = LLM(
        model=model_path,
        gpu_memory_utilization=GPU_MEM_UTIL,
        trust_remote_code=True,
        max_model_len=2048,
    )
    
    sampling_params = SamplingParams(
        temperature=0.0,
        top_p=1.0,
        max_tokens=max_tokens,
    )
    
    # GSM8K 스타일 프롬프트 (실제 평가와 유사한 워크로드)
    test_prompts = [
        [{"role": "user", "content": f"Solve this math problem step by step: What is {i*7 + 13} + {i*11 + 29}? Show your work."}]
        for i in range(num_prompts)
    ]
    
    # 워밍업
    print("[INFO] 워밍업...")
    _ = llm.chat(test_prompts[:2], sampling_params)
    
    # 실제 측정
    print(f"[INFO] 속도 측정 ({num_prompts} prompts, max_tokens={max_tokens})...")
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    start_time = time.time()
    outputs = llm.chat(test_prompts, sampling_params)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    elapsed = time.time() - start_time
    
    # 생성된 토큰 수 계산
    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    tokens_per_sec = total_tokens / elapsed
    time_per_token = elapsed / total_tokens
    
    print(f"  총 토큰: {total_tokens}")
    print(f"  소요 시간: {elapsed:.2f}초")
    print(f"  처리량: {tokens_per_sec:.1f} tokens/sec")
    print(f"  토큰당 시간: {time_per_token*1000:.2f} ms")
    
    # vLLM 정리
    del llm
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return {
        "total_tokens": total_tokens,
        "elapsed_sec": elapsed,
        "tokens_per_sec": tokens_per_sec,
        "time_per_token": time_per_token,
    }

print("\u2705 속도 측정 함수 정의 완료")

In [ ]:
# ============================================================================
# 원본 모델 속도 측정 (기준선)
# ============================================================================
print("=" * 60)
print("원본 모델 (FP16) 속도 측정")
print("=" * 60)

base_speed = measure_throughput(BASE_MODEL_PATH)
print(f"\n\u2705 베이스 속도: {base_speed['tokens_per_sec']:.1f} tok/s")

In [ ]:
# ============================================================================
# 양자화 모델 속도 측정
# ============================================================================
print("=" * 60)
print("양자화 모델 속도 측정")
print("=" * 60)

model_speed = measure_throughput(MODEL_PATH)
print(f"\n\u2705 모델 속도: {model_speed['tokens_per_sec']:.1f} tok/s")

In [ ]:
# ============================================================================
# SpeedNorm 계산
# ============================================================================
# SpeedNorm = 1 - (Time_model/Tokens_model) / (Time_base/Tokens_base)
#           = 1 - base_tok_per_sec / model_tok_per_sec

speed_norm = 1 - (base_speed['time_per_token'] / model_speed['time_per_token'])
# 동일한 공식의 다른 표현:
# speed_norm = 1 - (base_speed['tokens_per_sec'] / model_speed['tokens_per_sec'])  # 이건 틀림
# 올바른 공식: speed_norm = 1 - (model_time_per_token / base_time_per_token)
speed_norm = 1 - (model_speed['time_per_token'] / base_speed['time_per_token'])

print("=" * 60)
print("SpeedNorm 결과")
print("=" * 60)
print(f"  베이스 모델: {base_speed['tokens_per_sec']:.1f} tok/s ({base_speed['time_per_token']*1000:.2f} ms/tok)")
print(f"  양자화 모델: {model_speed['tokens_per_sec']:.1f} tok/s ({model_speed['time_per_token']*1000:.2f} ms/tok)")
print(f"  속도 향상: {model_speed['tokens_per_sec'] / base_speed['tokens_per_sec']:.2f}x")
print(f"  SpeedNorm: {speed_norm:.4f}")
print(f"\n\u26a0\ufe0f T4 기준 SpeedNorm입니다. L4에서는 다를 수 있습니다.")

---

# 5. 최종 점수 계산

In [ ]:
print("\n" + "=" * 60)
print("\U0001f3af 최종 점수 추정")
print("=" * 60)

if perf_norm is not None:
    estimated_score = max(0.5 * perf_norm + 0.5 * speed_norm, 0)
    
    print(f"\n  PerfNorm:  {perf_norm:.4f}  (GSM8K: {model_score:.4f} / {BASE_PERF})")
    print(f"  SpeedNorm: {speed_norm:.4f}  ({model_speed['tokens_per_sec']:.0f} vs {base_speed['tokens_per_sec']:.0f} tok/s)")
    print(f"")
    print(f"  Score = 0.5 × {perf_norm:.4f} + 0.5 × {speed_norm:.4f}")
    print(f"        = {0.5 * perf_norm:.4f} + {0.5 * speed_norm:.4f}")
    print(f"")
    print(f"  \u2b50 추정 Score: {estimated_score:.4f}")
    print(f"")
    print(f"  참고: V6 리더보드 점수 0.5955")
    if estimated_score > 0.5955:
        print(f"  \u2705 V6 대비 향상! (+{estimated_score - 0.5955:.4f})")
    else:
        print(f"  \u26a0\ufe0f V6 대비 하락 ({estimated_score - 0.5955:+.4f})")
else:
    print("\n  \u274c PerfNorm 미측정 - 위 셀에서 lm_eval 실행 필요")
    print(f"  SpeedNorm만: {speed_norm:.4f}")
    print(f"\n  PerfNorm을 수동 입력하려면 아래 셀 사용")

print("\n" + "=" * 60)

In [ ]:
# ============================================================================
# PerfNorm 수동 입력 (lm_eval 결과를 직접 입력)
# ============================================================================
MANUAL_SCORE = None  # 예: 0.5977 (GSM8K exact_match 값 입력)

if MANUAL_SCORE is not None:
    manual_perf_norm = MANUAL_SCORE / BASE_PERF
    manual_estimated = max(0.5 * manual_perf_norm + 0.5 * speed_norm, 0)
    
    print(f"수동 입력 GSM8K: {MANUAL_SCORE}")
    print(f"PerfNorm: {manual_perf_norm:.4f}")
    print(f"SpeedNorm: {speed_norm:.4f}")
    print(f"\u2b50 추정 Score: {manual_estimated:.4f}")

---

# 6. 여러 모델 비교 (선택)

여러 실험 결과를 한번에 비교할 때 사용.

In [ ]:
# ============================================================================
# 결과 기록 테이블
# 각 실험 결과를 여기에 추가하세요
# ============================================================================
ALL_RESULTS = {
    # "실험명": {"gsm8k": exact_match, "tok_per_sec": 속도, "zip_gb": zip크기},
    # 예시:
    # "V6 (baseline)": {"gsm8k": 0.5977, "tok_per_sec": 712, "zip_gb": 0.88},
    # "V13 (L0+L29)": {"gsm8k": 0.61, "tok_per_sec": 700, "zip_gb": 0.88},
    # "Exp B (damp 0.0008)": {"gsm8k": 0.62, "tok_per_sec": 700, "zip_gb": 0.88},
}

if ALL_RESULTS:
    # 베이스 속도 (이 세션에서 측정된 값 사용)
    base_tps = base_speed['tokens_per_sec']
    
    print("=" * 80)
    print("전체 실험 비교")
    print("=" * 80)
    print(f"{'실험':>20} | {'GSM8K':>7} | {'PerfNorm':>9} | {'tok/s':>7} | {'SpeedNorm':>10} | {'Score':>7}")
    print("-" * 80)
    
    for name, r in sorted(ALL_RESULTS.items(), key=lambda x: -(
        max(0.5 * x[1]["gsm8k"]/BASE_PERF + 0.5 * (1 - base_tps/x[1]["tok_per_sec"]), 0)
    )):
        pn = r["gsm8k"] / BASE_PERF
        sn = 1 - (1/r["tok_per_sec"]) / (1/base_tps)  # = 1 - base_tps/model_tps 가 아님
        sn = 1 - (r["tok_per_sec"] and (1/r["tok_per_sec"]) / (1/base_tps) or 0)
        # 정확한 공식: SpeedNorm = 1 - model_time_per_tok / base_time_per_tok
        #             = 1 - (base_tok_per_sec / model_tok_per_sec) ... 아님
        # SpeedNorm = 1 - (Time_model/Tokens_model) / (Time_base/Tokens_base)
        #           = 1 - time_per_tok_model / time_per_tok_base
        #           = 1 - (1/model_tps) / (1/base_tps)
        #           = 1 - base_tps / model_tps
        sn = 1 - base_tps / r["tok_per_sec"]
        sc = max(0.5 * pn + 0.5 * sn, 0)
        print(f"{name:>20} | {r['gsm8k']:>7.4f} | {pn:>9.4f} | {r['tok_per_sec']:>7.0f} | {sn:>10.4f} | {sc:>7.4f}")
    
    print("=" * 80)
else:
    print("[INFO] ALL_RESULTS에 결과를 추가하면 비교 표를 생성합니다")

---

# 빠른 점수 계산기

GSM8K 점수와 tok/s만 알면 바로 Score 추정.

In [ ]:
# ============================================================================
# ⭐ 빠른 점수 계산기
# ============================================================================
# 아래 값만 입력하면 바로 Score 추정

GSM8K_SCORE = 0.5977      # GSM8K exact_match (flexible-extract)
MODEL_TOK_PER_SEC = 712   # 양자화 모델 tokens/sec
BASE_TOK_PER_SEC = 461    # 원본 모델 tokens/sec (직접 측정했으면 그 값 사용)

# 계산
pn = GSM8K_SCORE / BASE_PERF
sn = 1 - BASE_TOK_PER_SEC / MODEL_TOK_PER_SEC
sc = max(0.5 * pn + 0.5 * sn, 0)

print(f"GSM8K: {GSM8K_SCORE} → PerfNorm: {pn:.4f}")
print(f"Speed: {MODEL_TOK_PER_SEC} tok/s (base: {BASE_TOK_PER_SEC}) → SpeedNorm: {sn:.4f}")
print(f"")
print(f"\u2b50 Score = 0.5×{pn:.4f} + 0.5×{sn:.4f} = {sc:.4f}")